# 📖 Module 03: Chunking Strategies

## GenAI L2 Exam Preparation

**Topics Covered:**
- Why chunking matters
- Fixed-size chunking
- RecursiveCharacterTextSplitter (⭐ default choice)
- Semantic chunking
- chunk_size and chunk_overlap tradeoffs
- Choosing the right strategy

**Source Material:** Class 35 (Chunking & Retriever)

---

## 1. Why Chunking Matters

### The Problem
Documents are often **too long** to:
1. Fit in an LLM's context window
2. Be embedded meaningfully as a single vector
3. Be retrieved with precision

### The Solution
Split documents into **smaller, semantically meaningful chunks** that can be individually embedded and retrieved.

### Impact on RAG Quality

```
Chunk too SMALL → Loses context, fragments meaning
Chunk too LARGE → Dilutes relevance, wastes token budget
Chunk just RIGHT → Precise retrieval, meaningful context ✅
```

### 🎯 Exam Tip (⭐ HIGH PRIORITY)
> Chunking tradeoffs are one of the **most commonly tested topics**.  
> Know what happens when chunk_size is too large vs too small.

In [ ]:
# Setup: Create a sample document for chunking demos
sample_text = """
Chapter 1: Introduction to Artificial Intelligence

Artificial Intelligence (AI) is the simulation of human intelligence processes by computer systems. These processes include learning (the acquisition of information and rules for using the information), reasoning (using rules to reach approximate or definite conclusions), and self-correction.

AI can be categorized into three types: Narrow AI (also known as Weak AI), General AI (also known as Strong AI), and Super AI. Narrow AI is designed to perform specific tasks, such as voice recognition or image classification. General AI would have the ability to understand, learn, and apply intelligence broadly, similar to human cognitive abilities. Super AI would surpass human intelligence in all aspects.

Chapter 2: Machine Learning Fundamentals

Machine Learning (ML) is a subset of artificial intelligence that provides systems the ability to automatically learn and improve from experience without being explicitly programmed. The learning process begins with observations or data, such as examples, direct experience, or instruction.

There are three main types of machine learning: supervised learning, unsupervised learning, and reinforcement learning. In supervised learning, the algorithm learns from labeled training data. In unsupervised learning, the algorithm identifies patterns in unlabeled data. Reinforcement learning involves an agent learning to make decisions by interacting with an environment.

Chapter 3: Deep Learning and Neural Networks

Deep Learning is a subset of machine learning that uses artificial neural networks with multiple layers (hence 'deep') to model and understand complex patterns in data. These deep neural networks are inspired by the structure and function of the human brain.

Key architectures in deep learning include Convolutional Neural Networks (CNNs) for image processing, Recurrent Neural Networks (RNNs) for sequential data, and Transformers for natural language processing. The Transformer architecture, introduced in the paper 'Attention is All You Need', has revolutionized NLP and forms the basis of modern large language models.
"""

print(f"📄 Sample text length: {len(sample_text)} characters")
print(f"📊 Approximate words: {len(sample_text.split())}")

## 2. Chunking Methods

### Overview

| Method | How it splits | Best For | Complexity |
|--------|--------------|----------|------------|
| **CharacterTextSplitter** | By single separator | Simple text | Low |
| **RecursiveCharacterTextSplitter** | By hierarchy of separators | Most use cases ⭐ | Medium |
| **TokenTextSplitter** | By token count | Token-aware splitting | Medium |
| **SemanticChunker** | By meaning/topic shifts | High-quality chunking | High |
| **MarkdownHeaderTextSplitter** | By markdown headers | Markdown docs | Low |
| **HTMLHeaderTextSplitter** | By HTML headers | Web pages | Low |

### 2.1 CharacterTextSplitter (Basic)

In [ ]:
from langchain.text_splitter import CharacterTextSplitter

# Splits on a single separator
splitter = CharacterTextSplitter(
    separator="\n\n",  # Split on double newlines (paragraphs)
    chunk_size=300,
    chunk_overlap=50,
    length_function=len
)

chunks = splitter.split_text(sample_text)

print(f"📊 Number of chunks: {len(chunks)}\n")
for i, chunk in enumerate(chunks):
    print(f"--- Chunk {i+1} ({len(chunk)} chars) ---")
    print(chunk[:150] + "..." if len(chunk) > 150 else chunk)
    print()

### 2.2 ⭐ RecursiveCharacterTextSplitter (DEFAULT CHOICE)

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Tries separators in order: \n\n → \n → . → " " → ""
splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " ", ""]  # Default hierarchy
)

chunks = splitter.split_text(sample_text)

print(f"📊 Number of chunks: {len(chunks)}\n")
for i, chunk in enumerate(chunks):
    print(f"--- Chunk {i+1} ({len(chunk)} chars) ---")
    print(chunk[:200] + "..." if len(chunk) > 200 else chunk)
    print()

### How RecursiveCharacterTextSplitter Works

```
Step 1: Try splitting by "\n\n" (paragraph breaks)
   → If chunk is still too large, go to step 2

Step 2: Try splitting by "\n" (line breaks)
   → If chunk is still too large, go to step 3

Step 3: Try splitting by ". " (sentences)
   → If chunk is still too large, go to step 4

Step 4: Try splitting by " " (words)
   → If chunk is still too large, go to step 5

Step 5: Split by character ("")
   → Last resort
```

**Why this is the default**: It preserves natural text boundaries (paragraphs > sentences > words) as much as possible.

## 3. ⭐ chunk_size and chunk_overlap Tradeoffs (EXAM CRITICAL!)

### chunk_size

| chunk_size | Pros | Cons |
|-----------|------|------|
| **Small** (100-300) | More precise retrieval, granular search | May lose context, fragments meaning |
| **Medium** (500-1000) ⭐ | Good balance of precision and context | — |
| **Large** (1500-3000) | More context per chunk | Dilutes relevance, wastes tokens, less precise retrieval |

### chunk_overlap

| chunk_overlap | Purpose | Typical Value |
|---------------|---------|---------------|
| **0** | No overlap — fastest, smallest storage | Risk of breaking context at boundaries |
| **10-20%** of chunk_size ⭐ | Preserves context at chunk boundaries | Recommended default |
| **50%+** | Maximum context preservation | Doubles storage, creates redundancy |

### 🎯 Exam Tip
> **Recommended defaults**: `chunk_size=1000`, `chunk_overlap=200`  
> **Remember**: overlap = 10-20% of chunk_size is the sweet spot

In [ ]:
# Demonstrate the effect of different chunk_size values
print("=" * 60)
print("EFFECT OF chunk_size ON CHUNKING")
print("=" * 60)

for size in [100, 300, 500, 1000]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=size,
        chunk_overlap=int(size * 0.2)  # 20% overlap
    )
    chunks = splitter.split_text(sample_text)
    avg_len = sum(len(c) for c in chunks) / len(chunks)
    
    print(f"\nchunk_size={size:>5} | overlap={int(size*0.2):>4} | chunks={len(chunks):>3} | avg_chunk_len={avg_len:.0f}")

In [ ]:
# Demonstrate chunk_overlap — showing the overlap between consecutive chunks
print("=" * 60)
print("VISUALIZING CHUNK OVERLAP")
print("=" * 60)

splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=50
)
chunks = splitter.split_text(sample_text)

# Show overlap between chunk 1 and chunk 2
if len(chunks) >= 2:
    chunk1_end = chunks[0][-60:]
    chunk2_start = chunks[1][:60]
    
    print(f"\n📝 Chunk 1 ends with:")
    print(f"   ...{chunk1_end}")
    print(f"\n📝 Chunk 2 starts with:")
    print(f"   {chunk2_start}...")
    print(f"\n🔄 The overlap ensures context is not lost at chunk boundaries!")

## 4. Using with Documents (not just text)

In [ ]:
from langchain_core.documents import Document
from langchain.text_splitter import RecursiveCharacterTextSplitter

# When you have Document objects (from loaders), use split_documents()
documents = [
    Document(page_content=sample_text, metadata={"source": "textbook.pdf", "page": 1})
]

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

# split_documents preserves and propagates metadata!
chunks = splitter.split_documents(documents)

print(f"📊 Input: {len(documents)} document → Output: {len(chunks)} chunks\n")
for i, chunk in enumerate(chunks):
    print(f"Chunk {i+1}: {len(chunk.page_content)} chars | metadata: {chunk.metadata}")

### 🎯 Exam Tip
> **`split_text()`** → takes a string, returns list of strings  
> **`split_documents()`** → takes list of Documents, returns list of Documents (metadata preserved!)  
> In a RAG pipeline, always use `split_documents()` to keep metadata.

## 5. Semantic Chunking (Advanced)

Unlike character-based splitting, semantic chunking splits text based on **meaning changes**.

### How it works:
1. Split text into sentences
2. Compute embeddings for each sentence
3. Measure similarity between consecutive sentences
4. Split where similarity drops significantly (topic change)

```python
from langchain_experimental.text_splitter import SemanticChunker
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

semantic_splitter = SemanticChunker(
    embeddings=embeddings,
    breakpoint_threshold_type="percentile"
)

chunks = semantic_splitter.split_text(text)
```

### When to Use Semantic Chunking
- Documents with **multiple topics** that flow into each other
- When fixed-size chunks **break logical groupings**
- When **retrieval quality** is more important than speed

### 🎯 Exam Tip
> Semantic chunking produces **variable-size chunks** based on content, not character count.  
> It's **slower** (requires embedding computation) but **higher quality**.

## 6. Chunking Strategy Decision Guide

```
What type of document?
│
├─ General text (articles, reports, books)
│  └─ RecursiveCharacterTextSplitter ⭐
│     chunk_size=1000, chunk_overlap=200
│
├─ Markdown documentation
│  └─ MarkdownHeaderTextSplitter
│     (splits by ## headers)
│
├─ Code files
│  └─ RecursiveCharacterTextSplitter.from_language()
│     (language-aware splitting)
│
├─ HTML content
│  └─ HTMLHeaderTextSplitter
│     (splits by <h1>, <h2>, etc.)
│
├─ Multi-topic documents where quality > speed
│  └─ SemanticChunker
│     (meaning-based splits)
│
└─ Token-critical applications
   └─ TokenTextSplitter
      (splits by exact token count)
```

## 🧠 Self-Assessment Quiz

---

**Q1.** What is the default text splitter recommended for most RAG pipelines?

<details>
<summary>Click for Answer</summary>

**RecursiveCharacterTextSplitter** — it tries multiple separators in order of preference (paragraphs → lines → sentences → words) to keep text as semantically coherent as possible.
</details>

---

**Q2.** If chunk_size is set too small (e.g., 50 characters), what problem occurs?

<details>
<summary>Click for Answer</summary>

Chunks become **too small to carry meaningful context**. A chunk like "the quick brown" is useless for retrieval because it has no complete thought or information. This leads to:  
- Poor embedding quality  
- Noisy retrieval results  
- LLM receiving fragmented, useless context
</details>

---

**Q3.** What is the purpose of chunk_overlap?

<details>
<summary>Click for Answer</summary>

Chunk overlap ensures that **context at chunk boundaries is not lost**. Without overlap, a sentence that spans two chunks would be cut and neither chunk would have the complete thought. Typical overlap is **10-20% of chunk_size**.
</details>

---

**Q4.** What is the difference between `split_text()` and `split_documents()`?

<details>
<summary>Click for Answer</summary>

- `split_text(string)` → takes a string, returns a list of strings  
- `split_documents(documents)` → takes a list of Document objects, returns a list of Document objects **with metadata preserved**  
In a RAG pipeline, use `split_documents()` to keep source metadata.
</details>

---

**Q5.** How does semantic chunking differ from recursive character splitting?

<details>
<summary>Click for Answer</summary>

- **Recursive Character**: Splits based on character count and separator hierarchy (fixed-size chunks)  
- **Semantic**: Splits based on **meaning changes** detected by embeddings (variable-size chunks)  
Semantic chunking is slower but produces higher-quality, more coherent chunks.
</details>

---

## ✅ Module 3 Complete!

**Key Takeaways:**
1. Chunking splits documents into manageable, searchable pieces
2. **RecursiveCharacterTextSplitter** is the go-to default
3. Recommended: `chunk_size=1000`, `chunk_overlap=200`
4. Too small = fragments meaning, too large = dilutes relevance
5. Semantic chunking gives better quality but is slower
6. Always use `split_documents()` (not `split_text()`) to preserve metadata

**Next:** [Module 04 — Embeddings Deep Dive](./04_Embeddings_Deep_Dive.ipynb)